In [4]:
!git clone https://github.com/gabriela708/ML-Household-Socioeconomic-Status

Cloning into 'ML-Household-Socioeconomic-Status'...
remote: Enumerating objects: 475, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 475 (delta 57), reused 48 (delta 48), pack-reused 391 (from 1)
Receiving objects: 100% (475/475), 9.03 MiB | 9.93 MiB/s, done.
Resolving deltas: 100% (228/228), done.


In [5]:
import os, sys
os.chdir("/content/ML-Household-Socioeconomic-Status")   # ajusta al nombre que te muestre !ls
sys.path.insert(0, "src")

In [7]:
!pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ses (pyproject.toml) ... done


In [8]:
!python scripts/make_target.py

Guardado data/processed/y_hogar.parquet  (19690 hogares)
            hogares   pct
ses                      
Pobre          4256  21.6
Vulnerable     4634  23.5
Media          9930  50.4
Alta            870   4.4


In [9]:
!python src/ses/features/build_features.py

Hogares GEIH: 24,159
Cabeceras (clase = 1): 21,037
Con cobertura en pobreza DANE: 19,690
Casos completos: 19,660
Excluidos por vacíos: 30
Sin fuga ✔ — guardado data/processed/X_hogar.parquet (19660, 33)


In [10]:
from ses.pipeline import cargar_entrenamiento

X,y,entrenamiento_cv= cargar_entrenamiento()

In [12]:
from ses.pipeline import  construir_preprocesador , BLOQUE_C

In [14]:
preprocesador=construir_preprocesador(BLOQUE_C)
preprocesador

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputar',
                                                  SimpleImputer(strategy='median')),
                                                 ('escalar',
                                                  StandardScaler())]),
                                 ['n_personas', 'edad_jefe', 'educ_jefe',
                                  'n_menores15', 'n_mayores65',
                                  'prop_dependientes', 'n_cuartos',
                                  'n_dormitorios', 'prop_afiliados_salud',
                                  'prop_subsidiado', 'n_ocupados',
                                  'n_cotizantes']),
                                ('nom',
                                 Pipeline(steps=[('imputar',...
                                  'combustible_cocina', 'tenencia_vivienda',
                                  'posicion_ocup_jefe']),
                                ('bin', SimpleImputer(strategy='most_frequent'),
                                 ['tiene_gas_natural', 'tiene_alcantarillado',
                                  'tiene_recoleccion_basuras',
                                  'tiene_acueducto', 'fin_corriente_o_cdt',
                                  'fin_cuenta_ahorros',
                                  'fin_credito_vivienda_vehiculo',
                                  'fin_prestamo_libre_inversion',
                                  'fin_tarjeta_credito',
                                  'fin_billetera_movil'])])

In [18]:
import numpy as np
import pandas as pd
from scipy.stats import chi2
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel

preprocesador.set_params(nom__onehot__min_frequency=30)   # categorías con < 30 hogares
Xp = preprocesador.fit_transform(X)
Xp = pd.DataFrame(Xp, columns=preprocesador.get_feature_names_out(), index=X.index)
Xp = Xp.drop(columns="nom__lugar_cocina_6.0")   # = "no cocina" de combustible_cocina , genera colinealidad
print(Xp.shape)                                # (15728, 73)
Xp.head()



(15728, 73)


,num__n_personas,num__edad_jefe,num__educ_jefe,num__n_menores15,num__n_mayores65,num__prop_dependientes,num__n_cuartos,num__n_dormitorios,num__prop_afiliados_salud,num__prop_subsidiado,...,bin__tiene_gas_natural,bin__tiene_alcantarillado,bin__tiene_recoleccion_basuras,bin__tiene_acueducto,bin__fin_corriente_o_cdt,bin__fin_cuenta_ahorros,bin__fin_credito_vivienda_vehiculo,bin__fin_prestamo_libre_inversion,bin__fin_tarjeta_credito,bin__fin_billetera_movil
hogar_id,,,,,,,,,,,,,,,,,,,,,
8436017_1_1,-0.533168,0.586091,-0.297465,-0.641716,-0.562960,-0.963838,0.622767,-1.043518,0.210273,1.174517,...,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
8436018_1_1,0.847302,1.304585,1.453611,-0.641716,1.005549,-0.183931,-0.303288,0.182939,0.210273,0.061473,...,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
8436019_1_1,-0.533168,1.364459,-0.297465,-0.641716,1.005549,0.595977,1.548822,-1.043518,0.210273,1.174517,...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
8436020_1_1,0.157067,1.244710,-1.698325,0.561683,1.005549,1.115915,-0.303288,0.182939,0.210273,1.174517,...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
8436021_1_1,0.157067,0.705840,-0.997895,-0.641716,1.005549,0.076038,-0.303288,0.182939,0.210273,1.174517,...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


Ajuste del POM

In [22]:
y_aligned = pd.Series(y.loc[Xp.index].values, index=Xp.index, name=y.name)
pom = OrderedModel(y_aligned, Xp, distr="logit")         # sin constante: los cortes hacen ese papel
res = pom.fit(method="bfgs", maxiter=5000, disp=False)
print("Convergió:", res.mle_retvals["converged"])
print(res.summary())

Convergió: True
                             OrderedModel Results                             
Dep. Variable:                    ses   Log-Likelihood:                -10973.
Model:                   OrderedModel   AIC:                         2.210e+04
Method:            Maximum Likelihood   BIC:                         2.268e+04
Date:                Tue, 27 Oct 2026                                         
Time:                        02:07:09                                         
No. Observations:               15728                                         
Df Residuals:                   15652                                         
Df Model:                          73                                         
                                                 coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------
num__n_personas                               -1.79

In [24]:
k = Xp.shape[1]
coef = res.params.iloc[:k]
ic = res.conf_int().iloc[:k]
tabla2 = pd.DataFrame({
    "coef": coef,
    "ee": res.bse.iloc[:k],
    "p": res.pvalues.iloc[:k],
    "OR": np.exp(coef),
    "OR_IC95_inf": np.exp(ic[0]),
    "OR_IC95_sup": np.exp(ic[1]),
}).round(4)
tabla2

,coef,ee,p,OR,OR_IC95_inf,OR_IC95_sup
num__n_personas,-1.7967,0.0494,0.0000,0.1658,0.1505,0.1827
num__edad_jefe,0.1926,0.0303,0.0000,1.2123,1.1424,1.2865
num__educ_jefe,0.4012,0.0261,0.0000,1.4936,1.4191,1.5719
num__n_menores15,0.2037,0.0469,0.0000,1.2259,1.1182,1.3441
num__n_mayores65,0.3515,0.0405,0.0000,1.4212,1.3128,1.5385
...,...,...,...,...,...,...
bin__fin_cuenta_ahorros,0.8390,0.0470,0.0000,2.3141,2.1102,2.5376
bin__fin_credito_vivienda_vehiculo,0.2496,0.1353,0.0652,1.2835,0.9844,1.6734
bin__fin_prestamo_libre_inversion,0.3155,0.0959,0.0010,1.3709,1.1361,1.6542
bin__fin_tarjeta_credito,0.5175,0.0754,0.0000,1.6779,1.4474,1.9450


Las variables nom__ siempre se comparan con su categoría de referencia, y eso hay que decirlo en el informe.

Para esta tabla nos permite observar las chances de cada coeficiente ademas de (usando el IC, si no contiene al 1) la suficiencia de este coeficiente, un OR alto o mayor a uno indica que van a la clase positiva

OR de IC enormes, como material_paredes_infrequent (0,34–10,5): son categorías con pocos hogares y no se interpretan.

In [26]:
cortes = res.model.transform_threshold_params(res.params)[1:-1]
n = np.bincount(y)
ll_nulo = (n * np.log(n / n.sum())).sum()        # modelo solo con cortes
lr = 2 * (res.llf - ll_nulo)
tabla3 = pd.Series({
    "corte Pobre | Vulnerable": cortes[0],
    "corte Vulnerable | Media": cortes[1],
    "corte Media | Alta": cortes[2],
    "log-verosimilitud": res.llf,
    "log-verosimilitud nulo": ll_nulo,
    "AIC": res.aic,
    "razón de verosimilitud (LR)": lr,
    "gl": k,
    "p LR": chi2.sf(lr, k),
}).round(4)
tabla3

,0
corte Pobre | Vulnerable,-3.2439
corte Vulnerable | Media,-1.0195
corte Media | Alta,5.2419
log-verosimilitud,-10972.9154
log-verosimilitud nulo,-18164.2025
AIC,22097.8308
razón de verosimilitud (LR),14382.5741
gl,73.0000
p LR,0.0000


Cortes: −3,24, −1,02 y 5,24 Son los umbrales o bordes de una escala latente de “nivel socioeconómico”. El hueco grande entre Media y Alta muestra que Alta es la clase más difícil de alcanzar.

Log-verosimilitud: −10.973 el modelo frente a −18.164 el nulo.
LR = 14.383, con 73 gl y p ≈ 0: los predictores en conjunto mejoran claramente al modelo nulo.



AIC = 22.098: Este nos permite cuando se comparen con otros modelos mirar cual es mejor (solo para eso)

In [31]:
Xc = sm.add_constant(Xp)
tabla4 = pd.DataFrame({"POM": coef})
fuera_ic = pd.Series(False, index=coef.index)
for j, nombre in enumerate(["corte 1 (≥ Vulnerable)", "corte 2 (≥ Media)", "corte 3 (Alta)"]):
    y_bin_values = (y.loc[Xc.index] > j).astype(int).values
    y_bin = pd.Series(y_bin_values, index=Xc.index)
    logit = sm.Logit(y_bin, Xc).fit(disp=False, maxiter=500)
    tabla4[nombre] = logit.params.drop("const")
    ic_j = logit.conf_int().drop("const")
    fuera_ic |= (coef < ic_j[0]) | (coef > ic_j[1])     # el coef. del POM no cabe en el IC de ese corte

cols = tabla4.columns[1:]
tabla4["cambia_signo"] = (tabla4[cols] > 0).nunique(axis=1) > 1
tabla4["no_estimable"] = (tabla4[cols].abs() > 10).any(axis=1)
tabla4["revisar"] = (tabla4["cambia_signo"] | fuera_ic) & ~tabla4["no_estimable"]
tabla4 = tabla4.round(3)
tabla4

/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


,POM,corte 1 (≥ Vulnerable),corte 2 (≥ Media),corte 3 (Alta),cambia_signo,no_estimable,revisar
num__n_personas,-1.797,-1.655,-2.328,-2.377,False,False,True
num__edad_jefe,0.193,0.013,0.161,0.579,False,False,True
num__educ_jefe,0.401,0.176,0.364,0.940,False,False,True
num__n_menores15,0.204,0.113,0.219,0.409,False,False,False
num__n_mayores65,0.351,0.374,0.476,0.094,False,False,True
...,...,...,...,...,...,...,...
bin__fin_cuenta_ahorros,0.839,0.794,0.824,0.764,False,False,False
bin__fin_credito_vivienda_vehiculo,0.250,0.326,0.156,0.348,False,False,False
bin__fin_prestamo_libre_inversion,0.315,0.515,0.319,0.429,False,False,False
bin__fin_tarjeta_credito,0.518,0.314,0.690,0.539,False,False,False


Me indica si en todsa las clases dependiendo del coeficiente se esta cumplioendo el supuesto de que en cada clase, las chances fueran constantes, en caso de no serlo es necesario revisar



Pero debido a que varias variables violan este supuesto con diferencias significativas entre OR, lo mejor para el siguiente paso es usar PPOM

In [32]:
tabla2.to_csv("reports/tables/tabla2_pom_coeficientes.csv")
tabla3.to_csv("reports/tables/tabla3_pom_ajuste.csv", header=["valor"])
tabla4.to_csv("reports/tables/tabla4_proporcionalidad.csv")